# Germany Data Centers + Estimations

Fetches all German data centers from the Data Center Database API, along with their estimations, into a list of dicts.

## Setup

In [ ]:
import requests

BASE_URL = "https://your-dashboard-domain.com"  # or http://localhost:3000 for local dev
API_KEY = "lm_your_api_key_here"

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
})

## Pagination helper

In [ ]:
def get_all_pages(path: str, params: dict | None = None) -> list[dict]:
    """Fetch every page of a v1 list endpoint and return the concatenated `data` rows."""
    params = dict(params or {})
    params.setdefault("per_page", 100)
    page = 1
    rows: list[dict] = []

    while True:
        params["page"] = page
        resp = session.get(f"{BASE_URL}{path}", params=params)
        resp.raise_for_status()
        body = resp.json()
        rows.extend(body["data"])

        pagination = body["pagination"]
        if pagination["page"] >= pagination["total_pages"]:
            break
        page += 1

    return rows

## Fetch all German data centers

In [ ]:
data_centers = get_all_pages("/api/v1/data-centers", {"country": "Germany"})
print(f"Found {len(data_centers)} data centers in Germany")

## Attach each data center's latest estimation

In [ ]:
def get_latest_estimation(data_center_id: str) -> dict | None:
    resp = session.get(
        f"{BASE_URL}/api/v1/data-centers/{data_center_id}/estimations",
        params={"latest": "true"},
    )
    if resp.status_code == 404:
        return None
    resp.raise_for_status()
    return resp.json()["data"]


records = []
for dc in data_centers:
    records.append({
        **dc,
        "latest_estimation": get_latest_estimation(dc["id"]),
    })

records[0]

## (Optional) Full estimation history instead of just the latest

In [ ]:
def get_estimation_history(data_center_id: str) -> list[dict]:
    return get_all_pages(f"/api/v1/data-centers/{data_center_id}/estimations")


for record in records:
    record["estimation_history"] = get_estimation_history(record["id"])

## (Optional) View as a DataFrame

In [ ]:
import pandas as pd

df = pd.json_normalize(records, sep="_")
df.head()